# Задача 9. Hand-crafted graph features

In [1]:
import numpy as np
import networkx as nx
import sklearn
from collections import defaultdict
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from networkx.generators import random_unlabeled_tree

## Генерация набора для классификации

In [2]:
dataset_len = 1500

In [6]:
default_tree_graphs = [random_unlabeled_tree(40) for _ in range(int(dataset_len * 0.5))]

In [7]:
balanced_tree_graphs = [nx.balanced_tree(r=2, h=5)  for _ in range(int(dataset_len * 0.5))]

In [8]:
labels = [0] * len(balanced_tree_graphs) + [1] * len(default_tree_graphs)
graphs = balanced_tree_graphs + default_tree_graphs

## Train/test split

In [9]:
train, test, y_train, y_test = sklearn.model_selection.train_test_split(graphs, labels, test_size=0.3, random_state=2, shuffle=True)

## Shortest Path Kernel

In [10]:
def shortest_path_kernel(
    train_graphs,
    test_graphs,
    max_path_length: int = 6,
    n_samples: int = 15):

    def compute_features(graphs):
        features = np.zeros((len(graphs), max_path_length))

        for i, graph in enumerate(graphs):
            valid_nodes = list(graph.nodes())

            for _ in range(n_samples):
                source, target = np.random.choice(valid_nodes, 2, replace=False)

                pl = nx.shortest_path_length(graph, source, target)

                pl = min(pl, max_path_length - 1)
                features[i, pl] += 1

        return features

    phi_train = compute_features(train_graphs)
    phi_test = compute_features(test_graphs)

    phi_train = phi_train / n_samples
    phi_test = phi_test / n_samples

    K_train = phi_train @ phi_train.T
    K_test = phi_test @ phi_train.T

    return K_train, K_test

## SVC Training

In [11]:
K_train, K_test = shortest_path_kernel(train, test)

model = SVC(kernel="precomputed", random_state=2).fit(K_train, y_train)
y_pred = model.predict(K_test)

In [12]:
accuracy_score(y_test, y_pred)

0.7066666666666667

In [13]:
f1_score(y_test, y_pred, average='weighted')

0.7059638840487882

## Подбор гиперпараметров

In [14]:
MAX_PATH_LENGTHS = [1, 10, 20, 30]
SAMPLES_COUNT = [1, 20, 40, 50]

In [15]:
def search_best_params(max_paths, samples_counts, train_graphs, test_graphs):
    for max_path in max_paths:
        for samples_count in samples_counts:

            K_train, K_test = shortest_path_kernel(train_graphs, test_graphs,max_path_length=max_path,n_samples=samples_count)

            model = SVC(kernel="precomputed", random_state=2)
            model.fit(K_train, y_train)
            y_pred = model.predict(K_test)

            print(f"Metrics for param MAX_PATH_LENGTHS={max_path} and SAMPLES_COUNT={samples_count}")
            print(f"Accuracy {accuracy_score(y_test, y_pred)}")
            print(f"F-1 Score {f1_score(y_test, y_pred, average='weighted')}")

In [16]:
search_best_params(MAX_PATH_LENGTHS, SAMPLES_COUNT, train, test)

Metrics for param MAX_PATH_LENGTHS=1 and SAMPLES_COUNT=1
Accuracy 0.5
F-1 Score 0.3333333333333333
Metrics for param MAX_PATH_LENGTHS=1 and SAMPLES_COUNT=20
Accuracy 0.5
F-1 Score 0.3333333333333333
Metrics for param MAX_PATH_LENGTHS=1 and SAMPLES_COUNT=40
Accuracy 0.5
F-1 Score 0.3333333333333333
Metrics for param MAX_PATH_LENGTHS=1 and SAMPLES_COUNT=50
Accuracy 0.5
F-1 Score 0.3333333333333333
Metrics for param MAX_PATH_LENGTHS=10 and SAMPLES_COUNT=1
Accuracy 0.5711111111111111
F-1 Score 0.5710581553278182
Metrics for param MAX_PATH_LENGTHS=10 and SAMPLES_COUNT=20
Accuracy 0.7755555555555556
F-1 Score 0.7752347795124894
Metrics for param MAX_PATH_LENGTHS=10 and SAMPLES_COUNT=40
Accuracy 0.8044444444444444
F-1 Score 0.8044096728307255
Metrics for param MAX_PATH_LENGTHS=10 and SAMPLES_COUNT=50
Accuracy 0.8822222222222222
F-1 Score 0.8822076799604889
Metrics for param MAX_PATH_LENGTHS=20 and SAMPLES_COUNT=1
Accuracy 0.5977777777777777
F-1 Score 0.5977281145820471
Metrics for param MAX_P

## Weisfeiler-Lehman Kernel

In [17]:
def compute_neighbor_cache(graph):
    return {node: tuple(nx.neighbors(graph, node)) for node in graph.nodes}

def compute_initial_colors(graph):
    return {node: 1 for node in graph.nodes}

def compute_feature_groups(graph, node_colors, neighbor_cache):
    feature_groups = defaultdict(list)
    
    for node in graph.nodes:
        neighbor_colors = tuple(sorted(node_colors[n] for n in neighbor_cache[node]))
        node_feature = (node_colors[node], neighbor_colors)
        feature_groups[node_feature].append(node)
    
    return feature_groups

def update_color_mapping(
    feature_groups,
    color_memo,
    unique_color_counter,
    limit
):
    new_colors = {}
    for features, nodes in feature_groups.items():
        if features not in color_memo:
            color_memo[features] = unique_color_counter
            unique_color_counter += 1
        
        mapped_color = color_memo[features] % limit
        for node in nodes:
            new_colors[node] = mapped_color
    
    return new_colors, color_memo, unique_color_counter

def wl_kernel_helper(
    graph,
    iterations: int = 3,
    limit: int = 1000
):
    neighbor_cache = compute_neighbor_cache(graph)
    node_colors = compute_initial_colors(graph)
    
    color_memo = {}
    unique_color_counter = 2 

    for _ in range(iterations):
        feature_groups = compute_feature_groups(graph, node_colors, neighbor_cache)
        
        node_colors, color_memo, unique_color_counter = update_color_mapping(
            feature_groups,
            color_memo,
            unique_color_counter,
            limit
        )

    colors = np.array(list(node_colors.values()), dtype=np.int32)
    histogram = np.bincount(colors, minlength=limit)
    return histogram / histogram.sum()

def wl_kernel(
    train_graphs,
    test_graphs,
    iterations: int = 3,
    limit: int = 1024
):
    phi_train = np.array([wl_kernel_helper(g, iterations, limit) for g in train_graphs])
    phi_test = np.array([wl_kernel_helper(g, iterations, limit) for g in test_graphs])

    K_train = phi_train @ phi_train.T
    K_test = phi_test @ phi_train.T

    return K_train, K_test

In [18]:
K_train_gk, K_test_gk = wl_kernel(train, test)
model = SVC(random_state=2)

model.fit(K_train_gk, y_train)
y_pred = model.predict(K_test_gk)

In [19]:
accuracy_score(y_test, y_pred)

1.0

In [20]:
f1_score(y_test, y_pred, average='weighted')

1.0

# Итог


- Сгенерирован набор данных для бинарной классификации графов (обычные и сбалансированные деревья)
- Реализована функция `shortest_path_kernel(train_graphs, test_graphs)`, которая принимает тренировочный и тестовые наборы, а возвращает пару `K_train, K_test`
- Используя реализованное ядро, обучена модель SVC. Подобраны гиперпараметры, рассчитаны различные метрики качества
- (+5 баллов) Также реализован Weisfeiler-Lehman Kernel. Обучен классификатор с ним